[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll use Chat Models, which take a sequence of messages as input and return messages as output. LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). By default, the course will use  [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) because it is both popular and performant. As noted, please ensure that you have an `OPENAI_API_KEY`.

Let's check that your `OPENAI_API_KEY` is set and, if not, you will be asked to enter it.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily
%pip install -U langchain-ollama


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it offers a good balance of quality, price, and speed, but you can also opt for the lower-priced `gpt-3.5` series or more recent models.

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [3]:
from langchain_ollama import ChatOllama

OLLAMA = "http://localhost:11434"
CHAT_MODEL = "qwen3:4b-instruct"

llm = ChatOllama(
    base_url=OLLAMA,
    model=CHAT_MODEL,
    temperature=0
)

print("Ollama successfully connected!")

Ollama successfully connected!


Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [6]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
llm.invoke(messages)

AIMessage(content='Hello! It\'s great to meet you. 😊  \n"Hello world" is a classic greeting—often the first line in programming tutorials!  \nHow can I assist you today? 🚀', additional_kwargs={}, response_metadata={'model': 'qwen3:4b-instruct', 'created_at': '2026-11-18T14:20:51.461141Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11308119800, 'load_duration': 6645109900, 'prompt_eval_count': 10, 'prompt_eval_duration': 441998000, 'eval_count': 41, 'eval_duration': 4205291000, 'logprobs': None, 'model_name': 'qwen3:4b-instruct', 'model_provider': 'ollama'}, id='lc_run--01a1ef07-f368-72ef-afd5-768102d2493e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 41, 'total_tokens': 51})

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [7]:
llm.invoke("hello world")

AIMessage(content='Hello! 👋  \n"Hello world" is a classic greeting — it\'s often the first line of a program in computer science, like in Python or C. It\'s a simple way to confirm that a computer is running and ready to execute code.  \n\nSo, you said "hello world" — that means you\'re either:  \n- Just saying hi, or  \n- Starting something fun like coding!  \n\nEither way, I\'m here to help. Whether you want to write a program, learn a language, or just chat — I\'m all in! 😊  \nWhat would you like to explore next? 🚀', additional_kwargs={}, response_metadata={'model': 'qwen3:4b-instruct', 'created_at': '2026-11-18T14:21:24.3523527Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16070922900, 'load_duration': 21519400, 'prompt_eval_count': 10, 'prompt_eval_duration': 362166000, 'eval_count': 128, 'eval_duration': 15671947000, 'logprobs': None, 'model_name': 'qwen3:4b-instruct', 'model_provider': 'ollama'}, id='lc_run--01a1ef08-6156-750b-a302-2d09103a6a70-0', tool_calls=[], inv

In [8]:
llm.invoke("hello world")

AIMessage(content='Hello! 👋  \n"Hello world" is a classic greeting — it\'s often the first line of a program in computer science, like in Python or C. It\'s a simple way to confirm that a computer is running and ready to execute code.  \n\nSo, you said "hello world" — that means you\'re either:  \n- Just saying hi, or  \n- Starting something fun like coding!  \n\nEither way, I\'m here to help. Whether you want to write a program, learn a language, or just chat — I\'m all in! 😊  \nWhat would you like to explore next? 🚀', additional_kwargs={}, response_metadata={'model': 'qwen3:4b-instruct', 'created_at': '2026-11-18T14:21:39.8170759Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15448780000, 'load_duration': 11912000, 'prompt_eval_count': 10, 'prompt_eval_duration': 155494000, 'eval_count': 128, 'eval_duration': 15264332000, 'logprobs': None, 'model_name': 'qwen3:4b-instruct', 'model_provider': 'ollama'}, id='lc_run--01a1ef08-a811-7393-a4f7-15fab2c38f9c-0', tool_calls=[], inv

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [11]:
_set_env("TAVILY_API_KEY")
%pip install -U langchain-tavily

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
import getpass

os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API Key: ")

In [19]:
from langchain_tavily import TavilySearch  # updated at 1.0
import os
import getpass

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY API Key: ")

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)
print(search_docs)

[{'url': 'https://www.ibm.com/think/topics/langgraph', 'title': 'What is LangGraph? | IBM', 'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. [...] LangGraph illuminates the processes within an AI workflow, allowing full transparency of the agent’s state. Within LangGraph, the “state” feature serves as a memory bank that records and tracks all the valuable information processed by the AI system. It’s similar to a digital notebook where the system captures and updates data as it moves through various stages of a workflow or graph analysis. [...] By using a graph-based

In [20]:
search_docs

[{'url': 'https://www.ibm.com/think/topics/langgraph',
  'title': 'What is LangGraph? | IBM',
  'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. [...] LangGraph illuminates the processes within an AI workflow, allowing full transparency of the agent’s state. Within LangGraph, the “state” feature serves as a memory bank that records and tracks all the valuable information processed by the AI system. It’s similar to a digital notebook where the system captures and updates data as it moves through various stages of a workflow or graph analysis. [...] By using a graph-b